In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/home/t/terencew/project-cluo/igvf/2023_YR2/multiome/'
samples = list(np.loadtxt(f'{projdir}/txt/samples.txt', dtype=str))
s = samples[0]

In [ ]:
adata = sc.read_h5ad(f'{projdir}/h5ad/rna/processed/intersect_atac_reindex_small.h5ad')

In [ ]:
adata

In [ ]:
justin_indir = '/u/project/cluo/Shared_Datasets/IGVF/202208_Pilot/new_multiome/YS3_Testrun3/YS3.v4_Metadata/'
meta_name = 'YS3.v4.metadata.txt'
justin_meta = pd.read_csv(f'{justin_indir}/{meta_name}', sep='\t', header=0, index_col=0)
adata.obs['clusters_v4'] = justin_meta.reindex(adata.obs.index)['Name']
adata.obs['coarse_v4'] = justin_meta.reindex(adata.obs.index)['Name']
adata.obs['coarse_v4'].fillna('NA', inplace=True)
adata.obs['coarse_v4'] = [x.split('_')[0] for x in adata.obs['coarse_v4']]
adata.obs['coarse_v4'].replace({'early1' : 'early', 
                              'early2' : 'early', 
                              'early3' : 'early', 
                              'early4' : 'early', 
                              'early5' : 'early', 
                              'ips1' : 'ips', 
                              'ips2' : 'ips', 
                              'ips3' : 'ips',
                              'ips4' : 'ips',
                              'ips5' : 'ips'}, inplace=True)

In [ ]:
sc.pl.umap(adata, color=['time_num', 'assignment'])

In [ ]:
wnn = pd.read_csv(f'{projdir}/csv/meta/joint/wnn_umap.csv', sep='\t', header=0, index_col=0)
wnn.shape

In [ ]:
adata.obsm['X_umap'] = wnn.values

In [ ]:
sc.pl.umap(adata, color=['time_num', 'assignment'])

In [ ]:
def cluster_small_multiples(adata, clust_key, size=10, ncols=2,
                            frameon=False, legend_loc=None, **kwargs):
    tmp = adata.copy()

    for i,clust in enumerate(adata.obs[clust_key].cat.categories):
        tmp.obs[clust] = adata.obs[clust_key].isin([clust]).astype('category')
        tmp.uns[clust+'_colors'] = ['#d3d3d3', adata.uns[clust_key+'_colors'][i]]

    sc.pl.umap(tmp, groups=tmp.obs[clust].cat.categories[1:].values,
               color=adata.obs[clust_key].cat.categories.tolist(),
               size=size, frameon=frameon, legend_loc=legend_loc,
               ncols=ncols, **kwargs)

In [ ]:
sc.pp.neighbors(adata, n_pcs=20, random_state=0)

In [ ]:
res = 1.4
sc.tl.leiden(adata, resolution=res, random_state=0)

In [ ]:
v5_clusts = pd.read_csv("/u/project/cluo/IGVF_Share/year2_pgp_dense/multiome/csv/joint/rna_leiden.csv",
                            sep="\t", header=0, index_col=0)

In [ ]:
adata.obs['v5_clusts'] = v5_clusts['leiden_labels']

In [ ]:
clusts = adata.obs['v5_clusts'].unique()

In [ ]:
clust = 'Fail1_4'
tmp_adata = adata[adata.obs['v5_clusts'] == clust]
tmp_adata.obs['leiden'].value_counts()

In [ ]:
adata.obs['tmp_time'] = adata.obs['time_num'] > 7
adata.obs['tmp_time'].replace({False : 'Before D7', True : 'After D7'}, inplace=True)
sc.pl.umap(adata, color=['leiden', 'tmp_time'], wspace=0.4)

In [ ]:
sc.pl.umap(adata, color=['leiden', 'time'], wspace=0.4, legend_loc='on data')

In [ ]:
clust = '0'
tmp_adata = adata[adata.obs['leiden'] == clust]
tmp_adata.obs['time'].value_counts()

In [ ]:
cluster_small_multiples(adata, 'leiden', ncols=6)

In [ ]:
rename_dict = {'0': 'Inter1_1',
                 '1': 'Start_1',
                 '2': 'IPS_2',
                 '3': 'Fail1_1',
                 '4': 'Fail1_2',
                 '5': 'Inter2_1',
                 '6': 'Inter2_2',
                 '7': 'Inter2_3',
                 '8': 'Inter2_4',
                 '9': 'IPS_1',
                 '10': 'IPS_3',
                 '11': 'Fail1_3',
                 '12': 'Fail2_1',
                 '13': 'Start_2',
                 '14': 'Inflamed',
                 '15': 'Inter1_2',
                 '16': 'Inter1_3',
                 '17': 'Fail2_2',
                 '18': 'Sendai_1',
                 '19': 'Fail1_4',
                 '20': 'Inter1_4',
                 '21': 'Inter1_5',
                 '22': 'Inter1_6',
                 '23': 'Sendai_2',
                 '24': 'Fail2_3',
                 '25': 'Inter1_7',
                 '26': 'Skin',
                 '27': 'Fail2_4',
                 '28': 'Inter1_8',
                 '29': 'Fail1_5'}

In [ ]:
# leiden = [str(x) for x in list(range(30))]
# labels = ['Inter1_1', 'Start_1', 'IPS_2', 'Fail1_1', 'Fail1_2', 'Inter2_1', 'Inter2_2', 'Inter2_3',
#          'Inter2_4', 'IPS_1', 'IPS_3', 'Fail1_3', 'Fail2_1', 'Start_2', 'Inter1_2', 'Inter1_3', 'Inter2_5',
#          'Sendai_1', 'Inflamed', 'Fail1_4', 'Inter1_4', 'Inter1_5', 'Inter1_6', 'Sendai_2', 'Fail2_2', 'Inter1_7',
#          'Skin', 'Fail2_3', 'Inter1_8', 'Fail1_5']
# rename_dict = dict(zip(leiden, labels))

In [ ]:
# leiden = [str(x) for x in list(range(30))]
# labels = ['Inter1_1', 'Start_1', 'IPS_2', 'Fail1_1', 'Fail1_2', 'Inter2_1', 'Inter2_2', 'Inter2_3',
#          'Inter2_4', 'IPS_1', 'IPS_3', 'Fail1_3', 'Fail2_1', 'Start_2', 'Inflamed', 'Inter1_2', 'Inter2_5',
#          'Sendai_1', 'Inter1_3', 'Fail1_4', 'Inter1_4', 'Inter1_5', 'Inter1_6', 'Sendai_2', 'Fail2_2', 'Inter1_7',
#          'Skin', 'Fail2_3', 'Inter1_8', 'Fail1_5']
# rename_dict = dict(zip(leiden, labels))

In [ ]:
adata.obs['v6_clusters'] = [rename_dict[x] for x in adata.obs['leiden']]
adata.obs['v6_clusters'].unique().shape

In [ ]:
sc.set_figure_params(figsize=(9,9), frameon=False)
sc.pl.umap(adata, color=['v6_clusters'], legend_loc='on data')

In [ ]:
fig, axes = plt.subplots(1, figsize=(16, 8))

# x_lab = 'donor'
# y_lab = 'value'
# hue = 'variable'

ax = sns.boxplot(data=adata.obs, x='v6_clusters', y='ambient_rna', hue='assignment', showfliers=False)
ax.set_xlabel('Cluster', fontsize=16)
ax.set_ylabel('Ambient RNA', fontsize=16)
ax.set_title('', fontsize=20)

ax.set_ylim(0, 0.5)
ax.tick_params(axis='x', labelsize=14, labelrotation=45)

sns.move_legend(ax, "upper left", bbox_to_anchor=(1.05, 1))

In [ ]:
sc.set_figure_params(figsize=(3,3), frameon=False)
cluster_small_multiples(adata, 'v6_clusters', ncols=6)

In [ ]:
adata.obs['traj_score'].head()

In [ ]:
clusts = adata.obs['v6_clusters'].unique()
traj_df = pd.DataFrame(index=clusts, columns=['traj_score'])
for clust in clusts:
    tmp_adata = adata[adata.obs['v6_clusters'] == clust]
    traj_df.loc[clust] = tmp_adata.obs['traj_score'].mean()
traj_df.sort_values(by='traj_score', inplace=True)
traj_order = traj_df.index

In [ ]:
traj_clusts = traj_df.index

In [ ]:
name_dict = {'Start_1' : 'newStart_1', 'Start_2' : 'newStart_2',
'Inter1_4' : 'newInter1_1', 'Fail1_4': 'newFail1_1',
'Inter1_3' : 'newInter1_2', 'Fail1_3' : 'newFail1_2',
'Inter1_7' : 'newInter1_3', 'Sendai_1' : 'newSendai_1',
'Fail1_1' : 'newFail1_3', 'Fail1_2' : 'newFail1_4',
'Skin' : 'newSkin', 'Sendai_2' : 'newSendai_2',
'Inter1_6' : 'newInter1_4', 'Inter1_5' : 'newInter1_5',
'Inter2_4' : 'newInter2_1', 'Inter1_2' : 'newInter1_6',
'Fail2_3' : 'newFail2_1', 'Inflamed' : 'newInflamed',
'Fail1_5' : 'newFail1_5', 'Inter1_8' : 'newInter1_7',
'Inter1_1' : 'newInter1_8', 'Inter2_3' : 'newInter2_2',
'Fail2_1' : 'newFail2_2', 'Inter2_2' : 'newInter2_3', 
'Fail2_2' : 'newFail2_3', 'Inter2_1' : 'newInter2_4',
'Fail2_4' : 'newFail2_4', 'IPS_1' : 'newIPS_1', 
'IPS_2' : 'newIPS_2', 'IPS_3' : 'newIPS_3'}

np.unique(list(name_dict.keys())).shape, \
np.unique(list(name_dict.values())).shape

In [ ]:
adata.obs['v6_clusters_traj'] = adata.obs['v6_clusters'].replace(name_dict)
adata.obs['v6_clusters_traj'] = [x.replace('new', '') for x in adata.obs['v6_clusters_traj']]

In [ ]:
clusts = adata.obs['v6_clusters_traj'].unique()
tmp_traj_df = pd.DataFrame(index=clusts, columns=['tmp_traj_score'])
for clust in clusts:
    tmp_adata = adata[adata.obs['v6_clusters_traj'] == clust]
    tmp_traj_df.loc[clust] = tmp_adata.obs['traj_score'].mean()
tmp_traj_df.sort_values(by='tmp_traj_score', inplace=True)

In [ ]:
# traj_df['clusts'] = traj_df.index
tmp_traj_df['tmp_clusts'] = tmp_traj_df.index
traj_df.reset_index(drop=True,inplace=True)
traj_df['clusts'] = traj_clusts
tmp_traj_df.reset_index(drop=True,inplace=True)

In [ ]:
merged_df = pd.concat([traj_df, tmp_traj_df], axis=1)
merged_df

In [ ]:
sorted(merged_df['clusts']) == sorted(merged_df['tmp_clusts'])

In [ ]:
merged_df[['traj_score', 'tmp_clusts']]

In [ ]:
merged_df['coarse'] = [x.split('_')[0] for x in merged_df['clusts']]
merged_df['tmp_coarse'] = [x.split('_')[0] for x in merged_df['tmp_clusts']]

In [ ]:
sum(merged_df['traj_score'] == merged_df['tmp_traj_score']), \
sum(merged_df['coarse'] == merged_df['tmp_coarse']), \
sorted(merged_df['clusts'].unique()) == sorted(merged_df['tmp_clusts'].unique())

In [ ]:
sc.set_figure_params(figsize=(9,9), frameon=False)
sc.pl.umap(adata, color=['v6_clusters_traj', 'time'], legend_loc='on data')

In [ ]:
adata.obs['v6_coarse'] = [x.split('_')[0] for x in adata.obs['v6_clusters_traj']]

In [ ]:
sc.set_figure_params(figsize=(9,9), frameon=False)
sc.pl.umap(adata, color=['v6_clusters_traj'], legend_loc='on data')

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.pl.umap(adata, color=['v6_coarse', 'time_num'], title=['Version 6', 'Time'])

In [ ]:
clusters = adata.obs.dropna()

In [ ]:
old_clusts = sorted(clusters['clusters_v4'].unique())
new_clusts = sorted(clusters['v6_clusters_traj'].unique())
clust_df = pd.DataFrame(index=new_clusts, columns=old_clusts)
for clust in new_clusts:
#     print(clust)
    tmp_clusts = clusters[clusters['v6_clusters_traj'] == clust]
    vals = tmp_clusts['clusters_v4'].value_counts().reindex(old_clusts).fillna(0)
    clust_df.loc[clust] = vals
### get rid of None column
clust_df = clust_df.iloc[:,1:]
clust_df = clust_df.astype(int)
clust_df.shape

In [ ]:
denom = clust_df.sum(axis=1)
clust_heatmap = clust_df.div(denom, axis=0).round(2)

In [ ]:
clust_heatmap.head()

In [ ]:
fig, axes = plt.subplots(1, figsize=(12,12))

ax = sns.heatmap(clust_heatmap, annot=False, vmin=0, vmax=1, cmap='Blues')
ax.grid(False)

In [ ]:
old_clusts = sorted(clusters['coarse_v4'].unique())
new_clusts = sorted(clusters['v6_coarse'].unique())
clust_df = pd.DataFrame(index=new_clusts, columns=old_clusts)
for clust in new_clusts:
#     print(clust)
    tmp_clusts = clusters[clusters['v6_coarse'] == clust]
    vals = tmp_clusts['coarse_v4'].value_counts().reindex(old_clusts).fillna(0)
    clust_df.loc[clust] = vals
### get rid of None column
# clust_df = clust_df.iloc[:,1:]
clust_df = clust_df.astype(int)
denom = clust_df.sum(axis=1)
clust_heatmap = clust_df.div(denom, axis=0).round(2)
clust_heatmap.shape

In [ ]:
fig, axes = plt.subplots(1, figsize=(20, 14))

ax = sns.heatmap(clust_heatmap, annot=True, cmap="Blues", linewidths=0, linecolor="none")
ax.grid(False)

In [ ]:
clusters.columns

In [ ]:
select = ['sample', 'assignment', 'time', 'time_num', 'donor_time',
          'leiden', 'traj_score', 'clusters_v4', 'coarse_v4', 'v6_clusters_traj', 'v6_coarse']
meta = adata.obs[select]
meta.rename(columns={'v6_clusters_traj' : 'clusters_v6', 'v6_coarse' : 'coarse_v6'}, inplace=True)
meta.shape

In [ ]:
meta.head()

In [ ]:
meta.to_csv(f'{projdir}/csv/clusters/rna_leiden_v6.csv', sep='\t', header=True, index=True)

### VERIFY!!!

In [ ]:
indir = '/u/project/cluo/IGVF_Share/year2_pgp_dense/multiome/csv/'
clusters = pd.read_csv(f'{indir}/rna_clusters_v6.csv', sep='\t', header=0, index_col=0)
clusters.shape

In [ ]:
adata.obs = clusters.copy()

In [ ]:
sc.set_figure_params(figsize=(9,9), frameon=False)
sc.pl.umap(adata, color=['coarse_v6', 'clusters_v6'], legend_loc='on data')

In [ ]:
!date